# Coding Self-Attention in PyTorch

We are only coding Encoder only model below. Encoder only model is the foundation for the modern models like BERT.

There are 3 main components:
1. Word Embeddings
2. Positional Encoding
3. Self-Attention

And the output: Context Aware Embeddings.

**At the heart of BERT is *Self-Attention***

In [4]:
import torch
from torch import nn # For the module, linear classes, and other helper functions
from torch.nn import functional as F # For Softmax

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model=2, row_dim=0,col_dim=1):
        ## d_model = the number of embedding values per token.
        ##           Because we want to be able to do the math by hand, we've
        ##           the default value for d_model=2.
        ##           However, in "Attention Is All You Need" d_model=512
        ##
        ## row_dim, col_dim = the indices we should use to access rows or columns
        super().__init__()
        # Next we need to create square matrices that contains the weights to create query matrix, key matrix, value matrix
        # Remember, the size of these square matrices is equal to the embedding size i.e. d_model here
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False) # We are saying that d_model num of columns are going to be your input and d_model num of columns should be in your output, based on that it will internally create weights (Query weights) and result in Query Matrix. Similarly for the rest of the items. Creating untrained weights here.
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        
        self.row_dim = row_dim
        self.col_dim = col_dim
        
    def forward(self, token_encodings):
        # Create query, key, and value matrices for the token encodings using their respective weight matrices
        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)
        
        # Now, we need to find the dot product of query and key (consider this to be finding the similarity scores between the query and key)
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
        
        # Scaling the dot product using square root of dim as mentioned in "Attention is all you need" paper; they found this giving better results
        # Bear in mind, this is not a proper scaling like MinMax or StandardScaling
        scaled_sims = sims / torch.tensor(k.size(self.col_dim)**0.5)
        
        # Apply softmax to the scaled_sims to get the weighted product of values, this will determine the final attention values
        attention_proportions = F.softmax(scaled_sims, dim=self.col_dim)
        attention_scores = torch.matmul(attention_proportions, v)
        
        return attention_scores

In [7]:
# Calculating Self-Attention

encoding_matrix = torch.tensor([[1.16, 0.23],
                                [0.57, 1.36],
                                [4.41, -2.16]])

torch.manual_seed(42)

self_attention = SelfAttention(d_model=2, row_dim=0, col_dim=1)

self_attention(encoding_matrix)

tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<MmBackward0>)

In [8]:
self_attention.W_q.weight.transpose(0, 1) # Weight matrix used for creating query matrix

tensor([[ 0.5406, -0.1657],
        [ 0.5869,  0.6496]], grad_fn=<TransposeBackward0>)

In [9]:
self_attention.W_k.weight.transpose(0, 1) # Weight matrix used for creating key matrix

tensor([[-0.1549, -0.3443],
        [ 0.1427,  0.4153]], grad_fn=<TransposeBackward0>)

In [10]:
self_attention.W_v.weight.transpose(0, 1) # Weight matrix used for creating value matrix

tensor([[ 0.6233,  0.6146],
        [-0.5188,  0.1323]], grad_fn=<TransposeBackward0>)

In [14]:
# Calculating Query Matrix

q = self_attention.W_q(encoding_matrix)

q

tensor([[ 0.7621, -0.0428],
        [ 1.1063,  0.7890],
        [ 1.1164, -2.1336]], grad_fn=<MmBackward0>)

In [15]:
# Calculating Key Matrix

k = self_attention.W_k(encoding_matrix)
k

tensor([[-0.1469, -0.3038],
        [ 0.1057,  0.3685],
        [-0.9914, -2.4152]], grad_fn=<MmBackward0>)

In [17]:
# Calculating Value Matrix

v = self_attention.W_v(encoding_matrix)
v

tensor([[ 0.6038,  0.7434],
        [-0.3502,  0.5303],
        [ 3.8695,  2.4246]], grad_fn=<MmBackward0>)

In [19]:
sims = torch.matmul(q, k.transpose(dim0=0, dim1=1))
sims

tensor([[-0.0990,  0.0648, -0.6523],
        [-0.4022,  0.4078, -3.0024],
        [ 0.4842, -0.6683,  4.0461]], grad_fn=<MmBackward0>)

In [21]:
scaled_sims = sims / (2**0.5)

scaled_sims

tensor([[-0.0700,  0.0458, -0.4612],
        [-0.2844,  0.2883, -2.1230],
        [ 0.3424, -0.4725,  2.8610]], grad_fn=<DivBackward0>)

To put everything in english:

* At first, we will split the input text into tokens.
* ADD positional encoding to the token encodings and get the new token encodings.
* Apply Self-Attention and get the contextual aware embeddings.

Now, we need to learn about each of these steps:
1. Token Encodings: The embeddings that come from the embedding layer during the transformer training process. We can also use the pre-trained embedding models like Word2Vec, or Glove embeddings. But in general, we use the embedding layer and it gets trained during the transformer training process.
2. Positional Encoding: We will add the cosine curve values at the given point or the sum of overlapped values of other curves (like cosine + sin or anything else) to get the positional encoding.
3. Self-Attention:
    * Before getting into the details of it, let's understand that in databases, we will have query to find the key and get its corresponding value. In our case, let's consider that it is not straight forward like a keyword matching; we won't get 100% matching and it is not always that a single value may not be the answer for our query. That means, Query can match with all the keys to some extent and all the values can be relevant to the given query via key to some extend. So, we will do this:
    * Find the dot product of query matrix with key matrix and divide this value by the square root of token encoding size. Consider this to finding the similarity between query and key. Now, apply softmax to scaled them to 0 to 1.
    * Now, we got some similarity scores between query and each key. Now, we want to find the corresponding values. So, we multiply this scaled dot product proportions with the values (since each value is relevant to some extent (be it +ve or -ve)) and get the contextually aware embeddings.

Note:
- At any given point, the Weights matrix for Query, Key and Value will be same for all the tokens in the input (the weights may change during the training phases after each epoch), but for the input sequence, they always remain the same for all the tokens in that epoch or after the training.
- That means, each token will have same query matrix, key matrix, and value matrix at the given point. To find the attention score of B w.r.t A, we will use the Q(Query matrix) of A and K and V of B.
- Since the token encodings + positional encodings will give a unique value to each token, the values Q, K, and V values for each word will be unique.
- Since the weight matrices for Query, Key, and Value will be same at any given point, we can do the required calculations for all the input tokens at once. This is the advantage of Transformers where it allows us to perform parallel operations and speed up things.
- Remember, we will be calculate the attention score for each token w.r.t to every other token (depending on masked or just self-attention) and take something like weighted sum, that's why we will get one single context aware embedding for each token.